# Book Recommendation Engine using KNN

In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
import os
if os.path.exists('data/BX-Books.csv'):
    books_filename = 'data/BX-Books.csv'
    ratings_filename = 'data/BX-Book-Ratings.csv'
else:
    if not os.path.exists('BX-Books.csv'):
        !wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
        !unzip -q -o book-crossings.zip
    books_filename = 'BX-Books.csv'
    ratings_filename = 'BX-Book-Ratings.csv'

In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding='ISO-8859-1',
    sep=';',
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding='ISO-8859-1',
    sep=';',
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

## Filtering, pivot table and KNN model

In [4]:
# Remove users with fewer than 200 ratings and books with fewer than
# 100 ratings (counts taken from the original dataset).
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()
df_ratings = df_ratings[df_ratings['user'].isin(user_counts[user_counts >= 200].index)]
df_ratings = df_ratings[df_ratings['isbn'].isin(book_counts[book_counts >= 100].index)]

df = df_ratings.merge(df_books, on='isbn', how='inner')
df = df.drop_duplicates(['user', 'title'])

book_user_matrix = df.pivot(index='title', columns='user', values='rating').fillna(0)
matrix = csr_matrix(book_user_matrix.values)

model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
model.fit(matrix)
print('matrix shape:', book_user_matrix.shape)

matrix shape: (673, 888)


In [5]:
# function to return recommended books - this will be tested
def get_recommends(book=''):
    distances, indices = model.kneighbors(
        book_user_matrix.loc[book].values.reshape(1, -1), n_neighbors=6)
    recommended_books = [
        [book_user_matrix.index[indices[0][i]], float(distances[0][i])]
        for i in range(5, 0, -1)
    ]
    return [book, recommended_books]

In [6]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016210794448853], ['The Weight of Water', 0.7708583474159241], ['The Surgeon', 0.7699410915374756], ['I Know This Much Is True', 0.7677075266838074], ['The Lovely Bones: A Novel', 0.7234864234924316]]]


In [7]:
# RUN THIS CELL TO TEST YOUR FUNCTION. DO NOT MODIFY CONTENTS.
def test_book_recommendation():
    test_pass = True
    recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
    if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
        test_pass = False
    recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
    recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
    for i in range(2):
        if recommends[1][i][0] not in recommended_books:
            test_pass = False
        if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
            test_pass = False
    if test_pass:
        print('You passed the challenge! 🎉🎉🎉🎉🎉')
    else:
        print("You haven't passed yet. Keep trying!")

test_book_recommendation()

You passed the challenge! 🎉🎉🎉🎉🎉
